# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}\n---\n{metadata['description']}")

## 2. Data Overview
Review available record sets, their fields, and IDs.

In [ ]:
# Helper to extract all record set, field, and column @ids from the Croissant metadata.
def extract_recordsets_fields_columns(metadata_dict):
    recordset_info = []
    # According to the Croissant schema, the main record sets appear under the 'recordSet' key.
    recordsets = metadata_dict.get('recordSet', [])
    # For this dataset, we need to reload the metadata with expanded details due to possible references
    # The actual recordSets may come from the resolved Croissant graph:
    # mlcroissant exposes dataset.record_sets for this purpose (with .to_json() for each).
    if hasattr(dataset, 'record_sets'):
        for rs in dataset.record_sets:
            rs_json = rs.to_json()
            rs_id = rs_json.get('@id', None)
            rs_name = rs_json.get('name', None)
            fields = rs_json.get('field', [])
            columns = rs_json.get('column', [])
            if not isinstance(fields, list):
                fields = [fields]
            if not isinstance(columns, list):
                columns = [columns]
            recordset_info.append({
                'record_set_id': rs_id,
                'name': rs_name,
                'fields': [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in fields],
                'columns': [c['@id'] if isinstance(c, dict) and '@id' in c else c for c in columns]
            })
        return recordset_info
    else:
        # fallback: not expected
        return []

overview = extract_recordsets_fields_columns(metadata)

if overview:
    for rs in overview:
        print(f"Record Set: {rs['name']}\n  @id: {rs['record_set_id']}")
        if rs['fields']:
            print("    Fields:")
            for f in rs['fields']:
                print(f"      - {f}")
        if rs['columns']:
            print("    Columns:")
            for c in rs['columns']:
                print(f"      - {c}")
        print()
else:
    print("No record sets listed in metadata. Please verify the schema.")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. All entities (record sets and fields) are referenced by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['record_set_id'] for rs in overview]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} rows for record set @id: {rs_id}")
    print(f"  Columns: {list(df.columns)}\n")

# Preview first record set loaded (if any)
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Sample from first record set ({first_rs_id}):")
    display(dataframes[first_rs_id].head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All columns are referenced by their `@id`.

In [ ]:
# As an example, select a numeric field for analysis from the first record set (if available)
import numpy as np

record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(record_set_id, pd.DataFrame())

numeric_field_id = None
# Find a numeric field: look for integer/float columns in the first record set
if not df.empty:
    for col in df.columns:
        # Attempt to coerce to numeric
        col_data = pd.to_numeric(df[col], errors='coerce')
        if col_data.notna().sum() > 0 and (col_data.dtype==float or col_data.dtype==int):
            numeric_field_id = col
            break

if numeric_field_id:
    # Convert column to numeric properly
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = np.nanmedian(df[numeric_field_id])  # median as sample threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric column
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by potential group field (try a likely categorical column)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < len(df) // 2:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric field found in the first record set for demonstration.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its grouping if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field (if exists)
if record_set_id and numeric_field_id and not df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Record Set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field exists, plot group-wise means
    if 'group_field' in locals() and group_field and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f"Mean of {numeric_field_id} grouped by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()


## 6. Conclusion

Through this notebook, we've loaded and explored the FAIR² dataset using `mlcroissant`, examined its tabular record sets, and illustrated simple processing and visualization steps referencing all entities by their `@id` fields. This approach provides a reproducible and FAIR-compliant data exploration workflow.